In [1]:
import pandas as pd
import numpy as np

In [61]:
results_session = pd.read_csv("da_cross_session_epho500.csv")

### Cross Session

In [62]:
results_session

,Unnamed: 0,align,no_align,time,distance,n_proj,n_epochs,seed,subject,target_subject,multifreq,reg,cross_subject
0,0,0.846975,0.822064,10.340035,les,1000,500,2473,1,0,False,10.0,False
1,1,0.846975,0.822064,10.340035,les,1000,500,2473,1,0,False,10.0,False
2,2,0.846975,0.822064,10.340035,les,1000,500,2473,1,0,False,10.0,False
3,3,0.846975,0.822064,10.340035,les,1000,500,2473,1,0,False,10.0,False
4,4,0.846975,0.822064,10.340035,les,1000,500,2473,1,0,False,10.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,0.768939,0.757576,5.798169,logsw,1000,500,929,9,0,False,1.0,False
496,496,0.768939,0.757576,5.798169,logsw,1000,500,929,9,0,False,1.0,False
497,497,0.768939,0.757576,5.798169,logsw,1000,500,929,9,0,False,1.0,False
498,498,0.768939,0.757576,5.798169,logsw,1000,500,929,9,0,False,1.0,False


In [ ]:
SPDSW-main/experiments/scripts/da_transfs_500.py

In [63]:
distances = results_session["distance"].unique()
subjects = results_session["subject"].unique()

In [64]:
L_session = {}
for distance in distances:
    L_session[distance] = {}
L_session["no_align"] = {}


reg = 10.0

for distance in distances:
    for s in subjects:
        dico_distances = {"mean_score": 0, "std_score": 0, "mean_time": 0, "std_time":0}
        
        if distance == "les":
            filt = (results_session["distance"] == distance) & (results_session["subject"] == s) & (results_session["reg"] == reg)
            
        else:
            filt = (results_session["distance"] == distance) & (results_session["subject"] == s)
            
        scores = results_session[filt]["align"]
        m_score = scores.mean()
        std_score = scores.std()
        
        times = results_session[filt]["time"]
        m_t = times.mean()
        std_t = times.std()
        
        dico_distances["mean_score"] = m_score
        dico_distances["std_score"] = std_score
        dico_distances["mean_time"] = m_t
        dico_distances["std_time"] = std_t
        
        L_session[distance][s] = dico_distances
        
        
for s in subjects:
    dico_distances = {"mean_score": 0, "std_score": 0, "mean_time": 0, "std_time":0}

    scores = results_session[
        (results_session["distance"] == distance)
        & (results_session["subject"] == s)
    ]["no_align"]
    m_score = scores.mean()
    std_score = scores.std()

    dico_distances["mean_score"] = m_score
    dico_distances["std_score"] = std_score

    L_session["no_align"][s] = dico_distances

In [65]:
acc_session = np.zeros((len(subjects), len(distances)+1))
acc_std_session = np.zeros((len(subjects), len(distances)))
t_session = np.zeros((len(subjects), len(distances)))
t_std_session = np.zeros((len(subjects), len(distances)))

for i, distance in enumerate(distances):
    for j, s in enumerate(subjects):
        acc_session[j, i] = L_session[distance][s]["mean_score"]
        acc_std_session[j, i] = L_session[distance][s]["std_score"]
        t_session[j, i] = L_session[distance][s]["mean_time"]
        t_std_session[j, i] = L_session[distance][s]["std_time"]
        
distance = "no_align"        
for j, s in enumerate(subjects):
    acc_session[j, -1] = L_session[distance][s]["mean_score"]

In [66]:
pd.DataFrame(acc_session, index=subjects, columns=np.concatenate([distances, ["no_align"]]))

,les,lew,spdsw,logsw,no_align
1,0.846975,0.843416,0.841993,0.841993,0.822064
3,0.860806,0.857143,0.861538,0.837363,0.798535
7,0.812274,0.812274,0.807220,0.791336,0.722022
8,0.830258,0.822878,0.835424,0.811070,0.793358
9,0.776515,0.776515,0.782576,0.779545,0.757576


In [67]:
pd.DataFrame(acc_session, index=subjects, columns=np.concatenate([distances, ["no_align"]])).mean(axis=0)

les         0.825366
lew         0.822445
spdsw       0.825750
logsw       0.812261
no_align    0.778711
dtype: float64

In [68]:
pd.DataFrame(acc_session, index=subjects, columns=np.concatenate([distances, ["no_align"]])).std(axis=0)

les         0.032805
lew         0.031052
spdsw       0.031006
logsw       0.027494
no_align    0.039203
dtype: float64

In [69]:
pd.DataFrame(acc_std_session, index=subjects, columns=distances)

,les,lew,spdsw,logsw
1,0.000000e+00,0.000000e+00,0.002906,0.006333
3,0.000000e+00,0.000000e+00,0.001495,0.009035
7,0.000000e+00,0.000000e+00,0.006834,0.007145
8,0.000000e+00,0.000000e+00,0.004519,0.009039
9,1.133117e-16,1.133117e-16,0.005244,0.007885


In [70]:
pd.DataFrame(t_session, index=subjects, columns=distances)

,les,lew,spdsw,logsw
1,10.029785,8.368496,5.995134,5.871681
3,9.889379,8.346414,5.889160,5.790722
7,9.967888,8.886570,5.933378,5.940041
8,9.938142,7.965588,5.772708,5.874333
9,13.542677,8.025240,5.743822,5.688416


In [71]:
pd.DataFrame(t_session, index=subjects, columns=distances).mean(axis=0)

les      10.673574
lew       8.318461
spdsw     5.866840
logsw     5.833038
dtype: float64

In [72]:
pd.DataFrame(t_session, index=subjects, columns=distances).std(axis=0)

les      1.604683
lew      0.366241
spdsw    0.106512
logsw    0.096636
dtype: float64

In [73]:
pd.DataFrame(t_std_session, index=subjects, columns=distances)

,les,lew,spdsw,logsw
1,0.237188,0.300876,0.101226,0.083794
3,0.148526,0.560068,0.113597,0.019972
7,0.185134,0.427015,0.105986,0.138013
8,0.184323,0.682995,0.026152,0.159599
9,0.304998,0.282138,0.145354,0.118519


### Cross Subjects

In [13]:
results_subject = pd.read_csv("/root/SPDSW-main/experiments/results/da_cross_subject_epho500.csv")

In [14]:
results_subject

,Unnamed: 0,align,no_align,time,distance,n_proj,n_epochs,seed,subject,target_subject,multifreq,reg,cross_subject
0,0,1.000000,1.000000,0.000000,spdsw,500,500,2473,1,1,False,1.0,True
1,1,0.681481,0.522222,5.682360,spdsw,500,500,2473,1,3,False,1.0,True
2,2,0.594096,0.505535,4.852123,spdsw,500,500,2473,1,7,False,1.0,True
3,3,0.681818,0.390152,4.802757,spdsw,500,500,2473,1,8,False,1.0,True
4,4,0.523207,0.265823,4.694083,spdsw,500,500,2473,1,9,False,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,245,0.454212,0.267399,5.097997,logsw,500,500,929,9,1,False,1.0,True
246,246,0.466667,0.285185,5.089938,logsw,500,500,929,9,3,False,1.0,True
247,247,0.324723,0.247232,5.064800,logsw,500,500,929,9,7,False,1.0,True
248,248,0.541667,0.397727,5.022394,logsw,500,500,929,9,8,False,1.0,True


In [15]:
distances = results_subject["distance"].unique()
subjects_src = results_subject["subject"].unique()
subjects_tgt = results_subject["target_subject"].unique()

In [16]:
L_subject = {}
for distance in distances:
    L_subject[distance] = {}
L_subject["no_align"] = {}


for distance in distances:
    for s1 in subjects_src:
        L_subject[distance][s1] = {}
        for s2 in subjects_tgt:
            if s1 != s2:
                L_subject[distance][s1][s2] = {}
        
                scores = results_subject[
                    (results_subject["distance"] == distance)
                    & (results_subject["subject"] == s1)
                    & (results_subject["target_subject"] == s2)
                ]["align"]
                m_score = scores.mean()
                std_score = scores.std()
                
                times = results_subject[
                    (results_subject["distance"] == distance)
                    & (results_subject["subject"] == s1)
                    & (results_subject["target_subject"] == s2)
                ]["time"]
                
                m_t = times.mean()
                std_t = times.std()
                
                L_subject[distance][s1][s2]["mean_score"] = m_score
                L_subject[distance][s1][s2]["std_score"] = std_score
                L_subject[distance][s1][s2]["mean_time"] = m_t
                L_subject[distance][s1][s2]["std_t"] = std_t
        
        
for s1 in subjects_src:
    L_subject["no_align"][s1] = {}
    for s2 in subjects_tgt:
        if s1 != s2:
            dico_distances = {"mean_score": 0, "std_score": 0, "mean_time": 0, "std_time":0}

            scores = results_subject[
                (results_subject["distance"] == distance)
                & (results_subject["subject"] == s1)
                & (results_subject["target_subject"] == s2)
            ]["no_align"]
            m_score = scores.mean()
            std_score = scores.std()

            dico_distances["mean_score"] = m_score
            dico_distances["std_score"] = std_score

            L_subject["no_align"][s1][s2] = dico_distances

In [17]:
results_mean = np.zeros((len(subjects_src), len(distances)+1))
t_subjects = np.zeros((len(subjects_src), len(distances)))

for l, distance in enumerate(L_subject):
    results = np.zeros((5,5))
    ts = np.zeros((5,5))
    for i, s1 in enumerate(subjects_src):
        for j, s2 in enumerate(subjects_tgt):
            if s1 != s2:
                results[i, j] = L_subject[distance][s1][s2]["mean_score"]
                if distance != "no_align":
                    ts[i, j] = L_subject[distance][s1][s2]["mean_time"]
                
        results_mean[i, l] = np.sum(results[i, :])/4
        
        if distance != "no_align":
            t_subjects[i, l] = np.sum(ts[i, :])/4
                       
    results_pd = pd.DataFrame(results, index=subjects_src, columns=subjects_tgt)
    print(distance)
    print(results_pd)
    print()

spdsw
          1         3         7         8         9
1  0.000000  0.680000  0.590406  0.687879  0.518143
3  0.684249  0.000000  0.710701  0.692424  0.568776
7  0.576557  0.697778  0.000000  0.608333  0.534177
8  0.627106  0.720741  0.538745  0.000000  0.556962
9  0.539194  0.590370  0.401476  0.601515  0.000000

logsw
          1         3         7         8         9
1  0.000000  0.686667  0.582288  0.658333  0.492827
3  0.698168  0.000000  0.697417  0.719697  0.545992
7  0.566300  0.671111  0.000000  0.568182  0.486076
8  0.630769  0.722222  0.501845  0.000000  0.552743
9  0.509158  0.563704  0.371956  0.580303  0.000000

no_align
          1         3         7         8         9
1  0.000000  0.522222  0.505535  0.390152  0.265823
3  0.344322  0.000000  0.309963  0.496212  0.274262
7  0.520147  0.533333  0.000000  0.261364  0.265823
8  0.498168  0.577778  0.243542  0.000000  0.396624
9  0.267399  0.285185  0.247232  0.397727  0.000000



In [18]:
pd.DataFrame(results_mean, index=subjects_src, columns=np.concatenate([distances, ["no_align"]]))

,spdsw,logsw,no_align
1,0.619107,0.605029,0.420933
3,0.664038,0.665319,0.356190
7,0.604211,0.572917,0.395167
8,0.610889,0.601895,0.429028
9,0.533139,0.506280,0.299386


In [19]:
pd.DataFrame(results_mean, index=subjects_src, columns=np.concatenate([distances, ["no_align"]])).mean(axis=0)

spdsw       0.606277
logsw       0.590288
no_align    0.380141
dtype: float64

In [20]:
pd.DataFrame(results_mean, index=subjects_src, columns=np.concatenate([distances, ["no_align"]])).std(axis=0)

spdsw       0.047105
logsw       0.057751
no_align    0.053326
dtype: float64

In [21]:
pd.DataFrame(t_subjects, index=subjects_src, columns=distances)

,spdsw,logsw
1,4.836450,4.769790
3,4.772561,4.899341
7,4.779487,4.861247
8,4.733225,4.815483
9,4.565292,4.646229


In [22]:
pd.DataFrame(t_subjects, index=subjects_src, columns=distances).mean(axis=0)

spdsw    4.737403
logsw    4.798418
dtype: float64

In [23]:
pd.DataFrame(t_subjects, index=subjects_src, columns=distances).std(axis=0)

spdsw    0.103025
logsw    0.097986
dtype: float64

### 